In [1]:
import pacmap
import numpy                            as np
import pandas                           as pd

import matplotlib.pyplot                as plt
import plotly.express                   as px
import plotly.graph_objects             as go

from tqdm                               import tqdm
from sklearn.preprocessing              import MinMaxScaler
from sklearn.mixture                    import GaussianMixture

from utils.Preprocessing2               import PreProcessData
from utils.plotFigures                  import lineAndScatterPlot, scatterPlot3D
from clustering.plotClustering          import ClusteringResults

In [2]:
preData = PreProcessData()
df, scaled_data, df_healthy, scaled_healthy = preData.preprocessing(type_file           = 'h5', 
                                                                    scaler              = MinMaxScaler(), 
                                                                    window_size         = 6, 
                                                                    time_aggregation    = '10min', 
                                                                    threshold           = 200)

# Manipulable

In [3]:
df_manipulable = scaled_data[preData.manipulated_cr1+preData.new_manipulable_cr1]

clusteringModels    = {'pacmap': GaussianMixture(n_components = 4, 
                                                 random_state = 2,  
                                                 n_init       = 5)} 

dimReductionModels  = {'pacmap': pacmap.PaCMAP(n_components = 3, 
                                               random_state = 40, 
                                               save_tree    = True)}

Manipulable = ClusteringResults(data                = scaled_data, 
                                dataNormalized      = df_manipulable, 
                                clusteringModels    = clusteringModels, 
                                dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
Manipulable.train()
Manipulable.getResults2()

# Calculate clusters for each trajectory
for traj in tqdm(Manipulable.trajectories):
    Manipulable.clustersResults[traj] = Manipulable.embedding[Manipulable.data['trajectory'] == traj]

c:\Users\CAEX\Desktop\Diego\Crusher Project\2ndrealdatajournalchile\mmodal\Lib\site-packages\pacmap\pacmap.py:828: UserWarning: Warning: random state is set to 40
  warnings.warn(f'Warning: random state is set to {_RANDOM_STATE}')
100%|██████████| 24/24 [00:00<00:00, 705.25it/s]


# Efficiency

In [4]:
df_efficiency   = scaled_data[preData.efficiency_cr1+preData.new_efficiency_cr1]

clusteringModels    = {'pacmap': GaussianMixture(n_components = 7, 
                                                 random_state = 3, 
                                                 n_init       = 5)} 

dimReductionModels  = {'pacmap': pacmap.PaCMAP(n_components = 3, 
                                               random_state = 40, 
                                               save_tree    = True)}

Efficiency = ClusteringResults(data                = scaled_data, 
                               dataNormalized      = df_efficiency, 
                               clusteringModels    = clusteringModels, 
                               dimReductionModels  = dimReductionModels)

# Train dimensionality reduction model
Efficiency.train()
Efficiency.getResults2()

# Calculate clusters for each trajectory
for traj in tqdm(Efficiency.trajectories):
    Efficiency.clustersResults[traj] = Efficiency.embedding[Efficiency.data['trajectory'] == traj]

c:\Users\CAEX\Desktop\Diego\Crusher Project\2ndrealdatajournalchile\mmodal\Lib\site-packages\pacmap\pacmap.py:828: UserWarning: Warning: random state is set to 40
  warnings.warn(f'Warning: random state is set to {_RANDOM_STATE}')
100%|██████████| 24/24 [00:00<00:00, 705.25it/s]


### Results processing

In [5]:
#OPTIONAL: Re-index the clusters to order the results display. Figures in the report use these indexes.
Manipulable.embedding['cluster_pacmap'] = Manipulable.embedding['cluster_pacmap'].astype('int').replace({0: '0', 2: '1', 3: '2', 1: '3'}).astype('str')

Efficiency.embedding['cluster_pacmap'] = Efficiency.embedding['cluster_pacmap'].astype('int').replace({3: '0', 1: '1', 4: '2', 6: '3', 2: '4', 0: '5', 5: '6'}).astype('str')

# Visualization

## Embedding colored by variables

In [6]:
scatterPlot3D(data  = Manipulable.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y',
              z     = f'pacmap_z', 
              color = preData.new_manipulable_cr1[1])

In [7]:
scatterPlot3D(data  = Efficiency.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              z     = f'pacmap_z',
              color = preData.efficiency_cr1[0])

## Clusters

In [8]:
scatterPlot3D(data  = Manipulable.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y',
              z     = f'pacmap_z', 
              color = 'cluster_pacmap')

In [9]:
scatterPlot3D(data  = Efficiency.embedding, 
              x     = f'pacmap_x', 
              y     = f'pacmap_y', 
              z     = f'pacmap_z',
              color = 'cluster_pacmap')

## Time serie colored by cluster

In [10]:
variable = preData.new_manipulable_cr1[1]
lineAndScatterPlot(Manipulable.embedding, x='tiempo', y=variable, color='cluster_pacmap')

In [11]:
variable = preData.efficiency_cr1[0]
lineAndScatterPlot(Efficiency.embedding, x='tiempo', y=variable, color='cluster_pacmap')

## Save results

In [12]:
df['pacmap_x_manipulable']  = Manipulable.embedding['pacmap_x']
df['pacmap_y_manipulable']  = Manipulable.embedding['pacmap_y']
df['cluster_manipulable']   = Manipulable.embedding['cluster_pacmap']

df['pacmap_x_efficiency']   = Efficiency.embedding['pacmap_x']
df['pacmap_y_efficiency']   = Efficiency.embedding['pacmap_y']
df['cluster_efficiency']    = Efficiency.embedding['cluster_pacmap']

In [15]:
df.to_hdf('data\processed_crusher.h5', key='crusher', mode='w')